# 06 - Model: K-Nearest Neighbors

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import NearestNeighbors

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"

tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
track_id_to_index = dict(zip(tracks["track_id"], tracks["content_index"]))
print(tracks.shape)

(89740, 40)


## Reuse 05's feature space

In [2]:
sound_matrix = np.load(MODEL_DIR / "cosine_feature_matrix.npy")
scaler = joblib.load(MODEL_DIR / "cosine_scaler.joblib")

with open(MODEL_DIR / "cosine_config.json", "r", encoding="utf-8") as file:
    cosine_config = json.load(file)

SOUND_FEATURES = cosine_config["features"]

## Pick a distance metric

In [3]:
def genre_match_rate(metric: str, sample_playlists: list[list[int]], k: int = 10) -> float:
    knn = NearestNeighbors(n_neighbors=k + 3, metric=metric, n_jobs=-1)
    knn.fit(sound_matrix)

    hits = []
    for playlist_indices in sample_playlists:
        playlist_genre = tracks.loc[playlist_indices[0], "track_genre"]
        centroid = sound_matrix[playlist_indices].mean(axis=0, keepdims=True)
        _, neighbor_indices = knn.kneighbors(centroid, n_neighbors=k + len(playlist_indices))
        recommended = [i for i in neighbor_indices[0] if i not in playlist_indices][:k]
        hits.append((tracks.loc[recommended, "track_genre"] == playlist_genre).mean())

    return float(np.mean(hits))


sample_genres = tracks["track_genre"].drop_duplicates().sample(8, random_state=3)
sample_playlists = [
    tracks[tracks["track_genre"] == genre].sample(3, random_state=3).index.tolist()
    for genre in sample_genres
]

for metric in ["euclidean", "manhattan", "cosine"]:
    print(f"{metric:10} {genre_match_rate(metric, sample_playlists):.3f}")

euclidean  0.013
manhattan  0.000


cosine     0.087


## Recommend

In [4]:
BEST_METRIC = "cosine"


def recommend_for_playlist(track_ids: list[str], top_n: int = 10, metric: str = BEST_METRIC) -> pd.DataFrame:
    indices = [track_id_to_index[tid] for tid in track_ids if tid in track_id_to_index]
    if not indices:
        raise ValueError("None of the given track_ids were found")

    weights = tracks.loc[indices, "popularity"].to_numpy() + 1
    centroid = np.average(sound_matrix[indices], axis=0, weights=weights).reshape(1, -1)

    playlist_genres = set(tracks.loc[indices, "track_genre"])
    candidate_mask = tracks["track_genre"].isin(playlist_genres)
    candidate_mask.iloc[indices] = False
    candidate_indices = tracks.index[candidate_mask].to_numpy()

    knn = NearestNeighbors(n_neighbors=min(top_n, len(candidate_indices)), metric=metric, n_jobs=-1)
    knn.fit(sound_matrix[candidate_indices])
    distances, neighbor_positions = knn.kneighbors(centroid)

    ranked_indices = candidate_indices[neighbor_positions[0]]
    columns = ["track_id", "track_name", "artists", "album_name", "track_genre", "popularity"]
    recommendations = tracks.loc[ranked_indices, columns].copy()
    recommendations["distance"] = distances[0]
    return recommendations.reset_index(drop=True)

In [5]:
demo_playlists = {
    genre: tracks[tracks["track_genre"] == genre].sample(3, random_state=5)["track_id"].tolist()
    for genre in tracks["track_genre"].drop_duplicates().sample(2, random_state=5)
}

for genre, playlist in demo_playlists.items():
    names = tracks[tracks["track_id"].isin(playlist)]["track_name"].tolist()
    print(f"\n{genre}: {names}")
    display(recommend_for_playlist(playlist, top_n=5))


funk: ['One Little Christmas Tree', 'Stephen King', 'Zika Sai Pra Lá']


,track_id,track_name,artists,album_name,track_genre,popularity,distance
0,5BP7nEs9ZwcQVF564vMTol,Tropa da Lacoste,Kyan;Mu540,Tropa da Lacoste,funk,48,0.090126
1,55PxC4e821VhBy5osd7xY3,Leste É a Zona,Menor MC;DJ Matt D;Rincon Sapiência,Trap Funk Star,funk,44,0.105296
2,4NHyYN4FiMFP594Ygf8Qik,Freestyle pra Faixa Rosa,BIN;WillsBife,Freestyle pra Faixa Rosa,funk,45,0.129459
3,1p4bk0gDn8ItJF182RwkMQ,Mec Mec 2,DJ Cayoo,Mec Mec 2,funk,47,0.135444
4,4QxHonLRXXSyNVooQCkkAp,Comemoração,Mc Kadu,B.I.G.O.D.E,funk,42,0.136738



grindcore: ['Severed', 'Oust - Odor Eliminator', 'Orbiting Flesh']


,track_id,track_name,artists,album_name,track_genre,popularity,distance
0,3Z7SiCIVXQYnXFt2IlXaOS,The Feast,Genghis Tron,Board Up the House,grindcore,14,0.034307
1,0dSkFhNrYiz7bDYnplYmIy,Running Through the Blood (Fear of God),Last Days Of Humanity,Horrific Compositions of Decomposition,grindcore,12,0.041273
2,4y1LorjiWYkHr2V39q7c1D,Preserved for Pleasure,Gorgasm,Destined to Violate,grindcore,14,0.043497
3,1CN3iKcEJbMUsuwS8D4ChF,Requiem,Gadget,The Funeral March,grindcore,11,0.044305
4,1hcBMT4ND8JiZ2oozF2mLi,H5N1,Gadget,The Funeral March,grindcore,13,0.044322


## Save

In [6]:
knn_config = {
    "model": "knn",
    "use_case": "playlist",
    "features_source": "cosine_feature_matrix.npy",
    "metric": BEST_METRIC,
    "candidate_pool": "union_of_playlist_genres",
    "centroid_weighting": "popularity_weighted",
}
with open(MODEL_DIR / "knn_config.json", "w", encoding="utf-8") as file:
    json.dump(knn_config, file, indent=2)